# Session 13 — Apache Spark: DataFrames API I
**Duration:** 80 minutes · **Level:** Introductory / Intermediate · **Language:** PySpark

**Learning outcomes:** define schemas and Rows; manipulate Columns and expressions; apply filters and conditional logic; cast data types; perform introductory aggregations.

## 0–10 min · Live discussion: Are you following me?
Quick review: What is a DataFrame? What is a schema? How do we load a CSV? What is the difference between transformation and action?

**Prediction:** Does `df.select(...)` execute a Spark job immediately?

## Setup: shared dataset from Session 12
Run both cells. Each notebook is self-contained.

In [ ]:
# Run once in a fresh environment if PySpark is missing:
# %pip install pyspark==3.5.6
from pyspark.sql import SparkSession, functions as F, types as T
from pathlib import Path
import tempfile, json, csv
spark = SparkSession.builder.master("local[*]").appName("SparkTeaching").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

In [ ]:
base = Path(tempfile.mkdtemp(prefix="spark_course_"))
records = [
 (1,"2026-01-05","Madrid","Books",2,15.0,"online"),
 (2,"2026-01-06","Barcelona","Electronics",1,199.0,"store"),
 (3,"2026-01-07","Madrid","Books",3,12.5,"store"),
 (4,"2026-01-08","Valencia","Home",2,35.0,"online"),
 (5,"2026-01-09","Sevilla","Electronics",1,89.0,"online"),
 (6,"2026-01-10","Barcelona","Home",4,22.0,"store"),
 (7,"2026-01-11","Madrid","Electronics",2,55.0,"online"),
 (8,"2026-01-12","Valencia","Books",1,18.0,"store"),
 (9,"2026-01-13","Sevilla","Home",3,25.0,"online"),
 (10,"2026-01-14","Madrid","Home",1,45.0,"store")
]
cols = ["order_id","order_date","city","category","quantity","unit_price","channel"]
with (base / "sales.csv").open("w",newline="",encoding="utf-8") as f:
 w=csv.writer(f);w.writerow(cols);w.writerows(records)
with (base / "sales.json").open("w",encoding="utf-8") as f:
 for r in records:f.write(json.dumps(dict(zip(cols,r)))+"\n")
print("Teaching data directory:",base)

In [ ]:
schema = T.StructType([T.StructField("order_id",T.IntegerType(),False),T.StructField("order_date",T.StringType(),True),T.StructField("city",T.StringType(),True),T.StructField("category",T.StringType(),True),T.StructField("quantity",T.IntegerType(),True),T.StructField("unit_price",T.DoubleType(),True),T.StructField("channel",T.StringType(),True)])
df = spark.read.option("header",True).schema(schema).csv(str(base / "sales.csv"))
df.printSchema()
df.show(5)

## 10–40 min · Lecture: Schemas, Rows, Columns and expressions
A **schema** defines field names, data types and nullability. A **Row** represents a record. A **Column** is a symbolic expression; most DataFrame transformations are lazy.

In [ ]:
print("Columns:",df.columns)
print("Schema:",df.schema.simpleString())
first = df.first()  # action
print("First Row:",first)
print("City from Row:",first["city"])

### Selecting, renaming and deriving columns
`select`, `alias`, `withColumn` and `drop` create new DataFrames; they do not modify the original.

In [ ]:
enriched = (df
 .withColumn("revenue",F.col("quantity")*F.col("unit_price"))
 .withColumn("order_date",F.to_date("order_date"))
 .withColumnRenamed("city","sales_city"))
enriched.select("order_id","sales_city","revenue","order_date").show()
print("Original columns unchanged:",df.columns)

### Expressions and SQL interoperability
Column expressions can be composed with Spark functions. SQL expressions are available via `expr`.

In [ ]:
df.select("order_id",(F.col("quantity")*F.col("unit_price")).alias("revenue"),F.expr("quantity * unit_price * 1.21").alias("illustrative_gross")).show(5)

### Nulls and data quality
`isNull`, `isNotNull`, `coalesce` and `fillna` are useful for missing values. SQL null comparisons do not behave like Python `None` comparisons.

In [ ]:
dirty = df.unionByName(spark.createDataFrame([(11,"2026-01-15","Madrid","Books",None,10.0,"online")],schema))
dirty.filter(F.col("quantity").isNull()).show()
dirty.fillna({"quantity":0}).filter(F.col("order_id")==11).show()

## 40–70 min · Lecture: Conditions, types and introduction to aggregations
Use `filter`/`where` for row predicates. Combine boolean Column expressions using `&`, `|`, `~` and **parentheses**; Python `and`/`or` do not work for Spark Columns.

In [ ]:
df.filter((F.col("city")=="Madrid") & (F.col("quantity")>=2)).show()
df.where(F.col("category").isin("Books","Home")).show(5)

### Conditional expressions with `when` / `otherwise`

In [ ]:
classified = df.withColumn("order_size",F.when(F.col("quantity")>=3,"large").when(F.col("quantity")==2,"medium").otherwise("small"))
classified.select("order_id","quantity","order_size").show()

### Types and casting
Dates, decimals, integers and strings have different semantics. Casting invalid values can yield null or raise an error depending on Spark SQL ANSI configuration.

In [ ]:
typed = df.withColumn("order_date",F.to_date("order_date","yyyy-MM-dd")).withColumn("unit_price_decimal",F.col("unit_price").cast(T.DecimalType(10,2)))
typed.printSchema()
typed.select("order_date","unit_price_decimal").show(3)

### Aggregations: count, sum, average and groupBy
`groupBy(...).agg(...)` groups rows before aggregating. Distinguish row counts from quantity sums and revenue sums.

In [ ]:
with_revenue = df.withColumn("revenue",F.col("quantity")*F.col("unit_price"))
summary = (with_revenue.groupBy("city").agg(F.count("*").alias("orders"),F.sum("quantity").alias("units"),F.round(F.sum("revenue"),2).alias("revenue"),F.round(F.avg("revenue"),2).alias("avg_order_value")).orderBy(F.desc("revenue")))
summary.show()

### Filtering after aggregation

In [ ]:
with_revenue.groupBy("category").agg(F.sum("revenue").alias("total_revenue")).filter(F.col("total_revenue")>100).orderBy(F.desc("total_revenue")).show()

### Spark SQL: equivalent operation

In [ ]:
with_revenue.createOrReplaceTempView("sales_view")
spark.sql("SELECT city, COUNT(*) AS orders, ROUND(SUM(revenue),2) AS revenue FROM sales_view GROUP BY city ORDER BY revenue DESC").show()

### Lazy execution and explain plans

In [ ]:
plan = with_revenue.filter(F.col("revenue")>50).select("city","revenue")
plan.explain(mode="formatted")
print("Rows returned:",plan.count())

## 70–80 min · Challenge: Did you get it?
**Student tasks (8 min):** (1) Create a `revenue` column. (2) Keep only `online` orders with revenue greater than 50. (3) Group by `city`, calculate total revenue and number of orders, sort descending. (4) Explain why `count()` is an action.

**Expected result:** Sevilla €164 (2 orders), Madrid €110 (1 order), Valencia €70 (1 order).

**Instructor discussion (2 min):** Compare the DataFrame API solution with a SQL equivalent.

In [ ]:
# TODO: Student challenge
# Start from df; write your transformation pipeline here.

### Instructor solution — reveal after the challenge

In [ ]:
result = (df.withColumn("revenue",F.col("quantity")*F.col("unit_price"))
 .filter((F.col("channel")=="online") & (F.col("revenue")>50))
 .groupBy("city").agg(F.count("*").alias("orders"),F.round(F.sum("revenue"),2).alias("total_revenue"))
 .orderBy(F.desc("total_revenue")))
result.show()
assert result.count()==3
assert abs(sum(r["total_revenue"] for r in result.collect())-344.0)<0.001

## Recap / exit ticket
1. What is the difference between a Row and a Column?
2. Why do we use `&` instead of `and`?
3. When would you use `cast`?
4. What is the difference between `sum(quantity)` and `count(*)`?
5. Why is `groupBy` a transformation but `show` an action?